# M41 · Relative relocation: differential times, cross-correlation and the double-difference method

**Week 5 · about 9.7 h** · before this: M21, M30, M31, M32

> **한국어 요약** 가까이 모인 지진들의 위치를 서로에 대해 정밀하게 다시 정하는 이중차분 상대 진원 재결정을 배웁니다. 도달 시간 차, 상호상관 지연 시간, 이중차분 방정식과 그 풀이를 함수로 직접 만들고, 상주 지진 여섯 개에서 HypoDD 결과와 비교합니다. 마지막으로 재실행과 부트스트랩으로 결과가 얼마나 재현되고 얼마나 확실한지 확인합니다.

**You will**

- Compute differential times from picks and by cross-correlation
- Set up and solve the double-difference equations
- Relocate the Sangju earthquakes and compare with HypoDD
- Find which run wrote a file, and bootstrap the errors

**New words:** relative relocation (상대 진원 재결정), differential time (도달 시간 차), double difference (이중차분), lag (지연 시간(시간 이동)), cycle skipping (주기 건너뜀, 사이클 스키핑), damping (감쇠 정규화, 댐핑), mean-shift constraint (평균 이동 구속 조건), bootstrap (부트스트랩) · [all terms](../../../docs/glossary.md)

**Optional reading:** Waldhauser & Ellsworth (2000), sections 1-2; Schaff et al. (2004), sub-sample delays

Log any AI help in `work/ai_log/M41.md` ([rules](../../../docs/ai_policy.md)); pass criteria: [criteria/M41.yaml](../../../.sgtl/criteria/M41.yaml).

---

© 2026 Minseong Seo (SGTL, Seoul National University) and contributors · code MIT · text CC BY 4.0 · cite CITATION.cff

In [ ]:
# Parameters: the only cell with values you may change. Run it first, then the setup cell.
DATASET = "sangju2019"                    # your STP download (M03) in the data folder
PQ_SLUG = "sangju_m21"                    # your PocketQuake cluster, located in M32
PQ_CORES = 3                              # cores per PocketQuake run (at most 4 on the shared server)
PQ_MEM_GB = 8.0                           # memory cap per run in GB (HypoDD needs at least 6)
VP_KM_S, VS_KM_S = 5.98, 3.40             # Part 6: P and S speeds of your straight rays (kim1983, 0-15 km)
MODEL_NAME = "kim1983"                    # Part 6: layered model for your station limits
CC_BAND_HZ = (5.0, 20.0)                  # Part 6: band-pass of the correlation windows (Hz)
CC_WINDOW_S = (0.5, 0.5)                  # Part 6: window before and after each pick (s)
MAX_LAG_S = 0.5                           # Part 6: largest lag searched (s)
DAMP = 0.01                               # Part 6: damping of your relocation
N_ITER = 8                                # Part 6: rounds of your relocation
BOOT_N = 200                              # Part 7: replicas per bootstrap
BOOT_SEED = 0                             # Part 7: seed of the first bootstrap
BOOT_SEED_CHANGED = 1000                  # Part 7: seed of the second bootstrap (the one change)
FIGURE_DIR = "figures"                    # your figures go here, next to this notebook
RESULT_DIR = "results"                    # your result files go here, next to this notebook

In [ ]:
import sgtl_tutorial as st
S = st.init("M41")

import contextlib
import hashlib
import io
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from obspy import UTCDateTime, read

Path(FIGURE_DIR).mkdir(exist_ok=True)
Path(RESULT_DIR).mkdir(exist_ok=True)
RUN_DIR = st.pq.pipeline_dir() / "pipeline" / "runs" / PQ_SLUG      # PocketQuake's results for your cluster
DD_DIR = RUN_DIR / "2.HypoDD"                                       # what its relocation stages write
RUNS_LOG = st.data.repo_path("work", ".sgtl", "runs.jsonl")         # sgtl's log of your PocketQuake runs
print("PocketQuake results of your cluster:", RUN_DIR)

## Part 0 · Absolute and relative locations (about 20 min)

Two nearby earthquakes send waves to a far station along almost the same path. An error of the velocity model then
cancels in the difference of their travel times. **Relative relocation** uses such differences to place nearby
events relative to each other. The standard program for it is HypoDD.

The six Sangju earthquakes of your download (KMA catalog, UTC):

In [ ]:
SAC_ROOT = st.data.root() / DATASET / "stp_download" / "SAC"
kma = st.data.sequence("sangju2019")
folders = sorted(p.name for p in SAC_ROOT.iterdir() if p.is_dir()) if SAC_ROOT.is_dir() else []
six = kma[kma["origin_utc"].dt.strftime("%Y%m%d%H%M%S").isin(folders)].reset_index(drop=True)
print(f"{len(folders)} earthquakes in your download {DATASET!r}; in the KMA catalog subset:")
print(six[["event_id", "origin_utc", "latitude", "longitude", "depth_km", "ml"]].to_string(index=False))

## Part 1 · Differential times from a catalog (about 20 min)

The travel time $T_i^k$ of event i to station k is its arrival time minus its origin time. Two events picked at
the same station with the same phase give a **differential time**:

$$dt_k^{ij} = T_i^k - T_j^k = (t_i^k - t_{0,i}) - (t_j^k - t_{0,j})$$

HypoDD's helper program ph2dt writes these times to `dt.ct`.

### M41-T1 · Catalog differential times · core · about 60 min

Write `catalog_dt(picks, origins, min_obs=1)`. It returns one row per event pair and station-phase picked for
both, with `dt_s` = travel time of `ev1` minus that of `ev2`. Pairs with fewer than `min_obs` rows are dropped.
See the docstring.

Example: origins `{"E1": 100.0, "E2": 400.0}` and P picks at `XX.AB01` at `105.20` and `405.05` s give one row:
`E1, E2, XX.AB01, P, 0.15`.

Not allowed: the course package.

In [ ]:
# Before you code: the catalog origin time of one event is 0.3 s too late. What happens to the differential times
# of its pairs?
st.predict("M41-T1", """

""")

In [ ]:
def catalog_dt(picks, origins, min_obs=1):
    """Catalog differential times of every event pair at the station-phases picked for both events.

    picks   : DataFrame, one row per pick, with at least the columns event_id (text), station (text), phase ("P"
              or "S") and time_s (arrival time, s after one reference moment); each event, station and phase at
              most once; any row order, any index; may have other columns; do not change it
    origins : dict event_id -> origin time (s after the same moment); do not change it
    min_obs : int >= 1; keep a pair when it has at least min_obs common station-phases
    returns : DataFrame with the columns ev1, ev2, station, phase, dt_s: one row per event pair and station-phase
              picked for both events (no NaN), each pair once, with ev1 the smaller id (compared as text);
              dt_s = (t_ev1 - t0_ev1) - (t_ev2 - t0_ev2) in s; rows sorted by ev1, ev2, station and phase,
              with a fresh index 0, 1, 2, ...
    """
    # Step 1: a travel time for every pick: its time minus the origin time of its event.
    # Step 2: pair the table with itself on station and phase; keep the pairs whose first id is the smaller one.
    # Step 3: dt_s = travel time of ev1 - travel time of ev2; drop pairs with fewer than min_obs rows; sort.
    # YOUR CODE HERE
    raise NotImplementedError("M41-T1: write your code here")

In [ ]:
st.check("M41-T1", func=catalog_dt)

Your function on a made-up catalog, then with E2's origin 0.3 s late, then with XX.AB02's clock 0.3 s late.

In [ ]:
practice = pd.DataFrame({
    "event_id": ["E1", "E1", "E1", "E2", "E2", "E2", "E3", "E3", "E3"],
    "station": ["XX.AB01", "XX.AB02", "XX.AB02", "XX.AB01", "XX.AB02", "XX.AB02", "XX.AB01", "XX.AB02", "XX.AB03"],
    "phase": ["P", "P", "S", "P", "P", "S", "P", "P", "P"],
    "time_s": [105.20, 107.80, 113.50, 405.05, 407.95, 413.70, 705.40, 707.60, 709.10]})
practice_origins = {"E1": 100.0, "E2": 400.0, "E3": 700.0}
as_given = catalog_dt(practice, practice_origins)
late_e2 = catalog_dt(practice, {**practice_origins, "E2": 400.3})
slow_clock = practice.assign(time_s=practice["time_s"] + 0.3 * (practice["station"] == "XX.AB02"))
compare = as_given.assign(**{"E2 origin 0.3 s late": late_e2["dt_s"],
                             "XX.AB02 clock 0.3 s late": catalog_dt(slow_clock, practice_origins)["dt_s"]})
print(compare.round(3).to_string(index=False))
print("\npairs kept with min_obs = 3:", catalog_dt(practice, practice_origins, 3)[["ev1", "ev2"]]
      .drop_duplicates().to_numpy().tolist())

**Explain:** Why did the late clock of XX.AB02 change no differential time, while the late origin of E2 did?

## Part 2 · Cross-correlation: a lag to a fraction of a sample (about 30 min)

Nearby events that break the same way leave similar waveforms. Slide one window along the other and compare the
shapes at each shift (cross-correlation, M22). The best shift is the **lag** of window b against window a.

At a shift of k samples, the **correlation coefficient** is

$$c_k = \frac{\sum_i x_i\, y_{i+k}}{\sqrt{\sum x^2 \sum y^2}}$$

with x and y the windows a and b minus their means. It is 1 for the same shape and -1 for an upside-down copy. A
positive lag means b is a later copy of a.

A parabola through the largest value $c_0$ and its neighbors $c_{-1}$, $c_{+1}$ refines the lag below one sample
(**sub-sample precision**). Its top lies

$$\delta = \frac{c_{-1} - c_{+1}}{2\,(c_{-1} - 2 c_0 + c_{+1})}$$

samples away from $c_0$.

**Watch out:** a ringing wave matches itself again one period later. With noise, the lag can jump by a whole period
(**cycle skipping**).

### M41-T2 · A lag from cross-correlation, to a fraction of a sample · core · about 120 min

Write `xcorr_lag(a, b, dt_s, max_lag_s)`. It returns `(lag_s, cc_peak, lags_s, cc)`: the lag refined by the
parabola, the largest $c_k$, and every whole lag up to `max_lag_s` with its $c_k$. See the docstring.

Example: `a = [0, 1, 0, 0, 0]`, `b = [0, 0, 1, 0, 0]`, `dt_s = 0.01` and `max_lag_s = 0.02` give
`cc = [-0.1, -0.3, -0.25, 0.95, -0.35]`, `cc_peak = 0.95` and `lag_s = 0.0098`.

Not allowed: library correlation and fitting tools (see the docstring) and the course package.

In [ ]:
# Before you code: b is a copy of a delayed by 0.0137 s, sampled at 100 Hz. At which whole-sample lag will the
# largest value sit?
st.predict("M41-T2", """

""")

In [ ]:
def xcorr_lag(a, b, dt_s, max_lag_s):
    """Lag of b against a by cross-correlation, refined below one sample by a parabola.

    a, b      : 1-D numpy arrays of the same length (at least 3); unchanged
    dt_s      : sampling interval (s)
    max_lag_s : largest lag searched (s): K = round(max_lag_s / dt_s) samples each way (round, not int():
                29.999999 must give 30), at least 1 and below the length
    returns   : (lag_s, cc_peak, lags_s, cc)
        lags_s  : 1-D array of the whole lags from -K to K samples, in s
        cc      : 1-D array of c_k (Part 2) at those lags, with the sum of x[i] y[i + k] over the samples where
                  both exist
        cc_peak : the largest value of cc (not of |cc|, not the parabola's top)
        lag_s   : the lag of cc_peak (s), moved to the parabola's top; positive when b is a later copy of a; at
                  the first or last lag, not moved
    Not allowed: numpy.correlate, numpy.convolve and numpy.polyfit (also those of numpy.ma), numpy.polynomial,
    scipy.signal, scipy.ndimage, scipy.interpolate, scipy.optimize, obspy.signal, torch, statsmodels, sgtl_tutorial.
    """
    # Step 1: x, y = a and b minus their means; K = round(max_lag_s / dt_s).
    # Step 2: for k = -K..K the sum of x[i] * y[i + k] over the overlap, divided by sqrt(sum x^2 * sum y^2).
    # Step 3: j = position of the largest value. Unless j is the first or last lag, fit the parabola through the
    #         values at j - 1, j, j + 1: its top lies (c[j-1] - c[j+1]) / (2 (c[j-1] - 2 c[j] + c[j+1])) samples
    #         from j.
    # YOUR CODE HERE
    raise NotImplementedError("M41-T2: write your code here")

In [ ]:
st.check("M41-T2", func=xcorr_lag)

Your function on an 8-Hz wavelet sampled at 100 Hz: a copy shifted by 0.0137 s, and an upside-down copy, whose
best match is a coefficient near -1.

In [ ]:
t_win = np.arange(-0.6, 0.6, 0.01)


def ricker(t, peak_hz):
    """A Ricker wavelet with its peak at t = 0."""
    arg = (np.pi * peak_hz * t) ** 2
    return (1.0 - 2.0 * arg) * np.exp(-arg)


wave_a = ricker(t_win, 8.0)
for label, wave_b in (("delayed copy", ricker(t_win - 0.0137, 8.0)),
                      ("upside-down copy", -ricker(t_win - 0.0137, 8.0))):
    lag_s, cc_peak, lags_s, cc = xcorr_lag(wave_a, wave_b, 0.01, 0.3)
    whole = lags_s[int(np.argmax(cc))]
    print(f"{label}: your lag {lag_s * 1000:.3f} ms (best whole sample {whole * 1000:.0f} ms), peak coefficient "
          f"{cc_peak:.4f}; the most negative coefficient {cc.min():.4f} at {lags_s[int(np.argmin(cc))] * 1000:.0f} ms")

lag_s, cc_peak, lags_s, cc = xcorr_lag(wave_a, ricker(t_win - 0.0137, 8.0), 0.01, 0.3)
fig, ax = plt.subplots(figsize=(6.0, 3.4))
ax.plot(lags_s * 1000.0, cc, "o-", color="tab:blue", ms=3, lw=1.0, label="Your correlation coefficients")
ax.axvline(lag_s * 1000.0, color="k", lw=1.0, label=f"Your lag ({lag_s * 1000:.2f} ms)")
ax.axvline(13.7, color="tab:red", lw=1.0, ls="--", label="True shift (13.7 ms)")
ax.set_xlim(-120, 150)
ax.set_xlabel("Lag of b against a (ms)")
ax.set_ylabel("Correlation coefficient")
ax.set_title("A delayed Ricker wavelet, 100 samples per second", fontsize=10)
fig.legend(loc="upper center", ncol=3, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.88))
fig.savefig(Path(FIGURE_DIR) / "correlation_function.png", dpi=150)

**Explain:** How close did your parabola get to the true shift of 13.7 ms, compared with the best whole sample?

In [ ]:
# Optional explorer E6 (not checked): your xcorr_lag on a wavelet and a shifted copy. Narrow the band and add
# noise: when does the lag jump by a whole period?
st.widgets.lag_explorer(xcorr=xcorr_lag)

## Part 3 · The double-difference system (about 30 min)

Use a local frame: x east, y north, z down, in km. Moving event i by $\Delta\mathbf{x}_i$ and its origin time by
$\Delta\tau_i$ moves its arrival at station k by about

$$\Delta t_i^k \approx \frac{\partial T_i^k}{\partial \mathbf{x}_i}\cdot\Delta\mathbf{x}_i + \Delta\tau_i$$

The **partial derivatives** say how fast the travel time changes as the source moves.

The **double difference** $r_k^{ij}$, the observed minus the predicted differential time, must be explained by the
moves of the two events:

$$\frac{\partial T_i^k}{\partial \mathbf{x}_i}\cdot\Delta\mathbf{x}_i + \Delta\tau_i
  - \frac{\partial T_j^k}{\partial \mathbf{x}_j}\cdot\Delta\mathbf{x}_j - \Delta\tau_j = r_k^{ij}$$

One row per observation gives $G\,m = d$: G holds the derivatives, m the four changes of each event, d the double
differences.

With one speed v, rays are straight, $T = |\mathbf{x}_i - \mathbf{s}_k| / v$, and the derivatives are the unit
vector from the station to the source, divided by the speed:

$$\frac{\partial T}{\partial \mathbf{x}_i} = \frac{\mathbf{x}_i - \mathbf{s}_k}{v\,|\mathbf{x}_i - \mathbf{s}_k|}$$

### M41-T3 · The double-difference system · core · about 120 min

Write `dd_system(events_km, stations_km, obs, vp, vs)`. It returns `(G, d)` for straight rays at the speed of each
row's phase. See the docstring for the column order.

Example: a station at `(0, 0, 0)`, events at `(0, 0, 10)` and `(0, 0, 12)`, `vp = 6` and one P row with
`dt_s = -0.30` give `G = [[0, 0, 1/6, 1, 0, 0, -1/6, -1]]` and `d = [0.0333]`.

Not allowed: numerical or automatic derivatives (the docstring lists the tools) and the course package.

In [ ]:
# Before you code: five events within 1 km, stations 15-40 km away. Which changes the differential times more:
# moving one event 100 m east, or the whole cluster?
st.predict("M41-T3", """

""")

In [ ]:
def dd_system(events_km, stations_km, obs, vp, vs):
    """The double-difference system G m = d for straight rays.

    events_km   : (N, 3) array of the events: x east, y north, z down (km); unchanged
    stations_km : (K, 3) array of the stations in the same frame, z down too; unchanged
    obs         : DataFrame with ev1, ev2 (row numbers of events_km), sta (row number of stations_km), phase
                  ("P" or "S") and dt_s (observed differential time, s); any index; unchanged
    vp, vs      : speeds (km/s) of the P rows and of the S rows
    returns     : (G, d), one row per row of obs, in its order
        G : (len(obs), 4N) array. Columns 4i to 4i+3 belong to event i: its changes of x, y, z (km) and origin
            time (s). A row holds, in ev1's columns, its dT/dx, dT/dy, dT/dz (s/km) at station sta and +1; in
            ev2's columns, minus the same of ev2; elsewhere 0.
        d : 1-D array, observed minus predicted: dt_s - (T_ev1 - T_ev2), in s
    Not allowed: scipy.optimize, scipy.differentiate, numdifftools, torch, jax, autograd and sgtl_tutorial.
    """
    # Step 1: for every row: v = vp or vs; r_i = event ev1 minus station sta, r_j = event ev2 minus station sta;
    #         their lengths R_i, R_j; predicted times R / v.
    # Step 2: event ev1's columns: r_i / (v R_i) and +1; event ev2's columns: -(r_j / (v R_j)) and -1.
    # Step 3: d = dt_s - (R_i - R_j) / v.
    # YOUR CODE HERE
    raise NotImplementedError("M41-T3: write your code here")

In [ ]:
st.check("M41-T3", func=dd_system)

Your function on a made-up cluster: how three moves change d, and the **singular values** of G. A change with a
zero singular value changes no differential time.

In [ ]:
toy_rng = np.random.default_rng(41)
toy_events = np.column_stack([toy_rng.uniform(-0.5, 0.5, 5), toy_rng.uniform(-0.5, 0.5, 5),
                              10.0 + toy_rng.uniform(-0.5, 0.5, 5)])
toy_az = np.radians(np.arange(8) * 45.0 + 10.0)
toy_dist = np.array([15.0, 22.0, 30.0, 40.0, 18.0, 26.0, 35.0, 20.0])
toy_stations = np.column_stack([toy_dist * np.sin(toy_az), toy_dist * np.cos(toy_az), np.zeros(8)])
toy_obs = pd.DataFrame([{"ev1": i, "ev2": j, "sta": k, "phase": ph, "dt_s": 0.0}
                        for i in range(5) for j in range(i + 1, 5) for k in range(8) for ph in ("P", "S")])
G_toy, _ = dd_system(toy_events, toy_stations, toy_obs, 6.0, 3.47)
moves = {"event 0 moved 100 m east": np.zeros(20), "every event moved 100 m east": np.zeros(20),
         "every origin time 0.1 s later": np.zeros(20)}
moves["event 0 moved 100 m east"][0] = 0.1
moves["every event moved 100 m east"][0::4] = 0.1
moves["every origin time 0.1 s later"][3::4] = 0.1
for label, change in moves.items():
    print(f"{label:>32}: largest change of a differential time {1000 * np.abs(G_toy @ change).max():8.3f} ms")
singular = np.linalg.svd(G_toy, compute_uv=False)
print(f"\nG has {G_toy.shape[0]} rows and {G_toy.shape[1]} columns; its singular values, largest first:")
print(np.array2string(singular, precision=4, max_line_width=110))

**Explain:** Which changes of the events did the differential times barely see, or not see at all?

## Part 4 · Solving: damping and a fixed centroid (about 30 min)

Least squares alone is unstable here: tiny errors in d can move the whole cluster by kilometers. Two remedies:

- **Damping** makes $|G\,m - d|^2 + \lambda^2 |m|^2$ smallest, so large changes cost something.
- The **mean-shift constraint** keeps the mean position of the events fixed: their changes add up to zero in x,
  y, z and origin time.

Damping adds the rows $\lambda I$ below G. The constraint adds four rows C (row c has a 1 in column 4e + c of every
event e) with a large weight w:

$$\begin{pmatrix} G \\ \lambda I \\ w\,C \end{pmatrix} m = \begin{pmatrix} d \\ 0 \\ 0 \end{pmatrix}$$

HypoDD's own weight w is only 1.

### M41-T4 · Damped least squares with a fixed centroid · core · about 90 min

Write `dd_solve(G, d, damp)`. It returns the least-squares m of the system above, with $\lambda$ = `damp`. Use
`numpy.linalg.lstsq` with $w = 10^4$ times the largest value of G.

Example: G = the identity of size 8 (two events), `d = [1, 0, 0, 0, 0, 0, 0, 0]` and `damp = 0` give
`m = [0.5, 0, 0, 0, -0.5, 0, 0, 0]`.

Not allowed: solvers that damp or constrain for you (see the docstring) and the course package.

In [ ]:
# Before you code: the true change also moves every event 300 m east. What will your solver do with those 300 m?
st.predict("M41-T4", """

""")

In [ ]:
def dd_solve(G, d, damp):
    """Damped least-squares solution of G m = d whose mean shift (x, y, z, origin time) over all events is zero.

    G    : (M, 4N) array, four columns per event in the order of dd_system; unchanged
    d    : (M,) array; unchanged
    damp : damping lambda >= 0 (the damping rows are damp times the identity, not damp squared)
    returns : (4N,) array m that makes |G m - d|^2 + damp^2 |m|^2 smallest among all m whose four mean shifts
              (the means over the events of their x, y, z and origin-time changes) are zero, to within 1e-6
              of its largest value
    Not allowed: solvers that damp or constrain for you (scipy.sparse.linalg, scipy.optimize, sklearn,
    statsmodels, torch, cvxpy) and sgtl_tutorial. Allowed: numpy.linalg and scipy.linalg.
    """
    # Step 1: n = number of columns; the identity of size n times damp, with n zeros on the right-hand side.
    # Step 2: four rows that add up the x, y, z and origin-time changes over the events, times a large weight
    #         (for example 1e4 times the largest |G|), with four zeros on the right-hand side.
    # Step 3: stack G, the damping rows and the four rows; solve the stacked system by least squares.
    # YOUR CODE HERE
    raise NotImplementedError("M41-T4: write your code here")

In [ ]:
st.check("M41-T4", func=dd_solve)

Your solver on the cluster of Part 3: a zero-mean change, the same change plus 300 m east, and more damping.

In [ ]:
true_change = np.column_stack([toy_rng.normal(0.0, 0.08, (5, 3)), toy_rng.normal(0.0, 0.01, 5)])
true_change -= true_change.mean(axis=0)                     # zero mean shift
shifted = true_change + np.array([0.3, 0.0, 0.0, 0.0])      # plus 300 m east for every event
for label, change in (("zero mean shift", true_change), ("plus 300 m east", shifted)):
    data = G_toy @ change.ravel()
    found = dd_solve(G_toy, data, 1e-6).reshape(-1, 4)
    shape_err = (found[:, :3] - found[:, :3].mean(0)) - (change[:, :3] - change[:, :3].mean(0))
    misfit_ms = 1000 * np.sqrt(np.mean((G_toy @ found.ravel() - data) ** 2))
    print(f"{label}: mean shift found (x, y, z in m) {np.round(1000 * found[:, :3].mean(0), 2)}; largest shape "
          f"error {1000 * np.abs(shape_err).max():.4f} m; RMS misfit {misfit_ms:.3f} ms")
data = G_toy @ true_change.ravel()
for lam in (0.0, 0.01, 0.1, 1.0):
    found = dd_solve(G_toy, data, lam)
    misfit_ms = 1000 * np.sqrt(np.mean((G_toy @ found - data) ** 2))
    print(f"damp {lam:5.2f}: length of the solution {np.linalg.norm(found):.4f} (true "
          f"{np.linalg.norm(true_change):.4f}), RMS misfit {misfit_ms:.3f} ms")

**Explain:** What did your solver do with the common 300 m east, and why did that cost almost no misfit?

## Part 5 · Rounds of linearized solutions (about 20 min)

The slopes in G change once the events move. So you solve in rounds (**iteration**): linearize, solve, move,
repeat. Before each round, subtract the origin shifts found so far from `dt_s`. An **observation weight**
multiplies its row of G and its d.

### M41-T5 · Iterate to convergence · core · about 90 min

Write `relocate(events_km, stations_km, obs, vp, vs, damp, n_iter, system, solve)`. It runs `n_iter` rounds with
the `system` and `solve` it is given (the check hands in its own). The docstring lists the steps of a round.

Example: with exact data, enough rounds bring the positions onto the true ones.

Not allowed: `scipy.optimize`, `torch` and the course package.

In [ ]:
# Before you code: a made-up cluster starts 300 m from the truth. How many rounds will it take until the RMS
# stops falling?
st.predict("M41-T5", """

""")

In [ ]:
def relocate(events_km, stations_km, obs, vp, vs, damp, n_iter, system, solve):
    """n_iter rounds of linearized double-difference solutions.

    events_km, stations_km, obs, vp, vs : as for dd_system (events_km are the start positions); obs may also have
              a column weight (>= 0; without it every weight is 1); change none of them
    damp    : damping for solve
    n_iter  : number of rounds (>= 1)
    system  : function system(positions, stations_km, obs, vp, vs) -> (G, d), like your dd_system
    solve   : function solve(G, d, damp) -> m, like your dd_solve
    A round: subtract the origin shifts found so far (shift of ev1 - shift of ev2) from dt_s; call system at the
    current positions; multiply each row of G and its value of d by the row's weight; call solve; add the
    changes to the positions (x, y, z) and to the origin shifts.
    returns : (positions_km, origin_shifts_s, rms_s)
        positions_km    : (N, 3) array, the final positions
        origin_shifts_s : (N,) array, the total origin shift of every event (s)
        rms_s           : n_iter + 1 values; rms_s[k] = sqrt(mean((weight * d)^2)) over all rows after k rounds,
                          so rms_s[0] is at the start positions
    """
    # Step 1: positions = a copy of events_km; shifts = zeros; w = the weight column or ones.
    # Step 2: each round: obs with dt_s - (shift[ev1] - shift[ev2]); G, d from system at the positions; record the
    #         RMS of w * d; m = solve(G * w per row, d * w, damp) as rows of four; add x, y, z and the origin part.
    # Step 3: after the last round, the system once more for the final RMS.
    # YOUR CODE HERE
    raise NotImplementedError("M41-T5: write your code here")

In [ ]:
st.check("M41-T5", func=relocate)

Your three functions on the cluster of Part 3, with exact data from the positions moved by `true_change`.

In [ ]:
truth_xyz = toy_events + true_change[:, :3]
exact = toy_obs.copy()
G_now, d_now = dd_system(truth_xyz, toy_stations, exact.assign(dt_s=0.0), 6.0, 3.47)
exact["dt_s"] = -d_now + (true_change[exact["ev1"], 3] - true_change[exact["ev2"], 3])   # exact differential times
for rounds in (1, 2, 3, 5):
    found, shifts, rms = relocate(toy_events, toy_stations, exact, 6.0, 3.47, 0.0, rounds, dd_system, dd_solve)
    print(f"{rounds} round(s): largest distance from the truth {1000 * np.abs(found - truth_xyz).max():9.4f} m, "
          f"origin shifts off by at most {1000 * np.abs(shifts - true_change[:, 3]).max():.4f} ms; "
          f"RMS by round (ms): {np.round(1000 * rms, 4).tolist()}")

In [ ]:
# Optional explorer E7 (not checked): your functions relocate a toy cluster under pick errors and station delays.
st.widgets.dd_toy(system=dd_system, solve=dd_solve, relocate=relocate)

## Part 6 · The Sangju earthquakes: HypoDD and your own functions (about 120 min)

You need your STP download (M03), the course tools and your cluster from M32 (`PQ_SLUG`). PocketQuake runs HypoDD
in four stages:

1. `ph2dt`: the catalog differential times, `dt.ct`;
2. `dtct`: HypoDD with `dt.ct`;
3. `rereference`, `xcorr`: cross-correlation times with a coefficient of 0.7 or more, `dt.cc`;
4. `dtcc`: HypoDD with both.

First, your HYPOINVERSE locations of M32 (picks with the default threshold):

In [ ]:
arc_files = sorted((RUN_DIR / "1.HypoInv" / "kim1983").glob("*.arc"))
if not arc_files:
    raise FileNotFoundError(f"PocketQuake has no HYPOINVERSE locations of cluster {PQ_SLUG!r} under {RUN_DIR}. Part 6 "
                            "starts from them: finish task M32-T1 of module M32 first (or set PQ_SLUG to the cluster "
                            "you located there).")
REGION = arc_files[0].stem                       # PocketQuake's name of your cluster in its file names
hypo = st.pq.read_sum(PQ_SLUG, "kim1983")
print(f"HYPOINVERSE (kim1983) of cluster {PQ_SLUG!r}, from {hypo.attrs['source']}:")
print(hypo[["id", "time", "lat", "lon", "depth", "num", "rms", "erh", "erz", "qual"]].to_string(index=False))
lowest = min(pd.read_csv(f)["Probability"].min() for f in sorted((RUN_DIR / "picks").glob("*_picks.csv")))
print(f"\nlowest pick probability in your cluster's pick files: {lowest:.3f} (PocketQuake's default threshold: 0.3)")
if lowest >= 0.5:
    print("Note: these picks come from a raised threshold. Rerun 'picking' to 'hypoinverse' with the default "
          "threshold first, as in module M32.")

A dry run first: read it, then run the next cell.

In [ ]:
st.pq.run(PQ_SLUG, "ph2dt", "dtct", cores=PQ_CORES, mem_gb=PQ_MEM_GB)          # dry run

In [ ]:
assert st.pq.run(PQ_SLUG, "ph2dt", "dtct", cores=PQ_CORES, mem_gb=PQ_MEM_GB, apply=True) == 0

The cross-correlation stage takes about 8-10 minutes on 3 cores. Never use more than 4 cores on the shared
server.

In [ ]:
assert st.pq.run(PQ_SLUG, "rereference", "xcorr", cores=PQ_CORES, mem_gb=PQ_MEM_GB, apply=True) == 0

In [ ]:
assert st.pq.run(PQ_SLUG, "dtcc", "dtcc", cores=PQ_CORES, mem_gb=PQ_MEM_GB, apply=True) == 0
COPIES = Path(RESULT_DIR) / "hypodd"


def copy_dtcc_products(folder):
    """Copy the products in the dt.cc folder that the last logged run wrote into results/hypodd/<folder>/."""
    last = json.loads(RUNS_LOG.read_text(encoding="utf-8").splitlines()[-1])
    copies = []
    for rel in last["products"]:
        if rel.startswith("2.HypoDD/02.dt.cc/"):
            copy = COPIES / folder / Path(rel).relative_to("2.HypoDD/02.dt.cc")
            copy.parent.mkdir(parents=True, exist_ok=True)
            copies.append(Path(shutil.copy2(RUN_DIR / rel, copy)))
    return copies


first_copies = copy_dtcc_products("first")
print("copied the dt.cc products of this run:", [str(p) for p in first_copies])
for line in RUNS_LOG.read_text(encoding="utf-8").splitlines()[-3:]:
    run = json.loads(line)
    print(run["ts"], f"{run['stage_from']} -> {run['through']}", f"{run['seconds']:.0f} s", run["products"])

Readers for HypoDD's files (not tasks):

In [ ]:
KM_PER_DEG = 6371.0 * np.pi / 180.0


def read_pha(path):
    """ph2dt's phase file: (picks, origins). '#' lines start an event (date, time, ..., id); 'STA TT WGHT PHA' lines
    are its picks, TT being the travel time from the HYPOINVERSE origin. Times in s after the first origin."""
    rows, starts, event = [], {}, None
    for line in Path(path).read_text().splitlines():
        part = line.split()
        if part and part[0] == "#":
            event = part[-1]
            starts[event] = UTCDateTime(*(int(v) for v in part[1:6])) + float(part[6])
        elif part:
            rows.append({"event_id": event, "station": part[0], "tt": float(part[1]), "weight": float(part[2]),
                         "phase": part[3]})
    zero = min(starts.values())
    origins = {e: t - zero for e, t in starts.items()}
    table = pd.DataFrame(rows)
    table["time_s"] = table["tt"] + table["event_id"].map(origins)
    return table, origins


def read_dt(path):
    """A dt.ct ('STA T1 T2 WGHT PHA') or dt.cc ('STA DT CC PHA') file: one row per observation, with its pair."""
    rows, pair = [], None
    for line in Path(path).read_text().splitlines():
        part = line.split()
        if part and part[0] == "#":
            pair = part[1:3]
        elif len(part) == 5:
            rows.append({"ev1": pair[0], "ev2": pair[1], "station": part[0], "dt_s": float(part[1]) - float(part[2]),
                         "weight": float(part[3]), "phase": part[4]})
        elif len(part) == 4:
            rows.append({"ev1": pair[0], "ev2": pair[1], "station": part[0], "dt_s": float(part[1]),
                         "cc": float(part[2]), "phase": part[3]})
    return pd.DataFrame(rows)


def read_table(path, names):
    """A whitespace-separated table without a header (event.dat, station.dat)."""
    return pd.read_csv(path, sep=r"\s+", header=None, names=names, usecols=range(len(names)), dtype={"id": str})


def flat_km(lat, lon, lat0, lon0):
    """East and north (km) of points from (lat0, lon0) on a flat local map (6371-km sphere)."""
    east = (np.asarray(lon, dtype=float) - lon0) * KM_PER_DEG * np.cos(np.radians(lat0))
    return east, (np.asarray(lat, dtype=float) - lat0) * KM_PER_DEG


events_dat = read_table(DD_DIR / "00.ph2dt" / "event.dat",
                        ["date", "hhmmss", "lat", "lon", "depth_km", "mag", "eh", "ez", "rms", "id"])
stations_dat = read_table(DD_DIR / "00.ph2dt" / "station.dat", ["station", "lat", "lon"])
print(f"event.dat: {len(events_dat)} events; station.dat: {len(stations_dat)} stations; ph2dt.inp settings:")
print("\n".join((DD_DIR / "00.ph2dt" / "ph2dt.inp").read_text().splitlines()[-2:]))

### Your catalog differential times against ph2dt

Your `catalog_dt` on ph2dt's phase file (without picks of negative weight, as MINWGHT = 0 says), against `dt.ct`:

In [ ]:
pha_picks, pha_origins = read_pha(DD_DIR / "00.ph2dt" / f"{REGION}_mod.pha")
used = pha_picks[pha_picks["weight"] >= 0.0]
mine_ct = catalog_dt(used, pha_origins)
ct = read_dt(DD_DIR / "00.ph2dt" / "dt.ct")
both_ct = ct.merge(mine_ct, on=["ev1", "ev2", "station", "phase"], how="left", suffixes=("_ph2dt", "_yours"))
print(f"phase file: {len(pha_picks)} picks of {len(pha_origins)} events, {len(used)} with a weight of 0 or more")
print(f"dt.ct: {len(ct)} differential times of {ct[['ev1', 'ev2']].drop_duplicates().shape[0]} pairs; yours: "
      f"{len(mine_ct)} of {mine_ct[['ev1', 'ev2']].drop_duplicates().shape[0]} pairs")
print(f"rows of dt.ct missing from yours: {both_ct['dt_s_yours'].isna().sum()}; largest |ph2dt - yours|: "
      f"{np.nanmax(np.abs(both_ct['dt_s_ph2dt'] - both_ct['dt_s_yours'])):.2e} s")

extra = mine_ct.merge(ct, on=["ev1", "ev2", "station", "phase"], how="left", indicator=True)
extra = extra[extra["_merge"] == "left_only"].copy()
ex, ny = flat_km(events_dat["lat"], events_dat["lon"], events_dat["lat"].mean(), events_dat["lon"].mean())
place = pd.DataFrame({"x": ex, "y": ny, "z": events_dat["depth_km"].to_numpy()}, index=events_dat["id"])
gap = place.loc[extra["ev1"]].to_numpy() - place.loc[extra["ev2"]].to_numpy()
extra["separation_km"] = np.linalg.norm(gap, axis=1)
# ph2dt.f: "remove outliers above the separation-delaytime line": |dt| > separation / v + 0.5 s, v = 4 (P), 2.3 (S)
extra["line_s"] = extra["separation_km"] / np.where(extra["phase"] == "P", 4.0, 2.3) + 0.5
print(f"\nyour rows that ph2dt left out: {len(extra)}; of them above ph2dt's separation-delay line: "
      f"{int((extra['dt_s_x'].abs() > extra['line_s']).sum())}")
print(extra[["ev1", "ev2", "station", "phase", "dt_s_x", "separation_km", "line_s"]]
      .rename(columns={"dt_s_x": "your dt_s"}).round(3).head(8).to_string(index=False))

The rows ph2dt left out are **outliers**: events less than 10 km apart cannot differ in travel time by several
seconds. M32 found the cause: an S wave picked as P.

### Your cross-correlation times against PocketQuake's

The cell cuts a window around each pick on both records, filters both the same way and runs your `xcorr_lag`. The
same filter on both windows does not move the lag (unlike a pick, M20). With window starts $s_a$, $s_b$ and lag
$\ell$:

$$dt = (s_a - t_{0,a}) - (s_b - t_{0,b}) - \ell$$

In [ ]:
cc_pq = read_dt(DD_DIR / "02.dt.cc" / "dt.cc_0.7_combined")
names = sorted(p.name for p in (RUN_DIR / "waveforms_100km").iterdir() if p.is_dir())
folder_of = {}
for row in events_dat.itertuples():                       # the folder whose name is nearest to the origin time
    when = UTCDateTime.strptime(f"{row.date}{int(row.hhmmss):08d}", "%Y%m%d%H%M%S%f")
    folder_of[row.id] = min(names, key=lambda f: abs(UTCDateTime.strptime(f, "%Y%m%d%H%M%S") - when))
cache = {}


def filtered(event, station, comp):
    """The re-referenced record of one event, station (NETSTA) and component, demeaned, tapered and band-passed."""
    key = (event, station, comp)
    if key not in cache:
        folder = folder_of[event]
        files = sorted((RUN_DIR / "waveforms_100km" / folder).glob(f"{folder}.{station[:2]}.{station[2:]}.*{comp}.sac"))
        trace = read(str(files[0]))[0] if files else None
        if trace is not None:
            trace.detrend("demean").taper(0.05)
            trace.filter("bandpass", freqmin=CC_BAND_HZ[0], freqmax=CC_BAND_HZ[1], corners=4, zerophase=True)
        cache[key] = trace
    return cache[key]


def cut(trace, center, before, after):
    """Samples from `before` s ahead of `center` to `after` s past it (a fixed count), and the first sample's time."""
    first = int(round((center - before - trace.stats.starttime) * trace.stats.sampling_rate))
    count = int(round((before + after) * trace.stats.sampling_rate)) + 1
    return trace.data[first:first + count].astype(float), trace.stats.starttime + first * trace.stats.delta


rows = []
for obs_row in cc_pq.itertuples():
    best = None
    for comp in (("Z",) if obs_row.phase == "P" else ("N", "E")):
        tr1, tr2 = filtered(obs_row.ev1, obs_row.station, comp), filtered(obs_row.ev2, obs_row.station, comp)
        if tr1 is None or tr2 is None:
            continue
        mark = "a" if obs_row.phase == "P" else "t0"
        t01 = tr1.stats.starttime - tr1.stats.sac.b         # the reference (origin) time of each record
        t02 = tr2.stats.starttime - tr2.stats.sac.b
        win1, start1 = cut(tr1, t01 + tr1.stats.sac[mark], *CC_WINDOW_S)
        win2, start2 = cut(tr2, t02 + tr2.stats.sac[mark], *CC_WINDOW_S)
        lag_s, cc_peak, _, _ = xcorr_lag(win1, win2, tr1.stats.delta, MAX_LAG_S)
        if best is None or cc_peak > best[1]:
            best = ((start1 - t01) - (start2 - t02) - lag_s, cc_peak)
    if best is not None:
        rows.append({"ev1": obs_row.ev1, "ev2": obs_row.ev2, "station": obs_row.station, "phase": obs_row.phase,
                     "pq_dt_s": obs_row.dt_s, "pq_cc": obs_row.cc, "your_dt_s": best[0], "your_cc": best[1]})
mine_cc = pd.DataFrame(rows)
mine_cc["diff_ms"] = 1000.0 * (mine_cc["your_dt_s"] - mine_cc["pq_dt_s"])
mine_cc["your cc"] = pd.cut(mine_cc["your_cc"], [-1.0, 0.7, 0.8, 0.9, 1.0], labels=["< 0.7", "0.7-0.8", "0.8-0.9",
                                                                                     ">= 0.9"])
summary_cc = mine_cc.groupby(["phase", "your cc"], observed=True)["diff_ms"].agg(
    times="size", median_ms="median", mad_ms=lambda v: (v - v.median()).abs().median(),
    over_10_ms=lambda v: int((v.abs() > 10.0).sum()))
print(f"dt.cc: {len(cc_pq)} times (coefficient >= 0.7 in PocketQuake's measurement); yours for {len(mine_cc)} of them")
print(summary_cc.round(2).to_string())
big = mine_cc[mine_cc["diff_ms"].abs() > 10.0]
print(f"\ndifferences larger than 10 ms: {len(big)}; their median size {big['diff_ms'].abs().median():.0f} ms")
mine_cc.to_csv(Path(RESULT_DIR) / "M41_cc_times.csv", index=False)

fig, ax = plt.subplots(figsize=(6.4, 3.8))
for phase, color in (("P", "tab:blue"), ("S", "tab:red")):
    part = mine_cc[mine_cc["phase"] == phase]
    ax.plot(part["your_cc"], part["diff_ms"], "o", ms=3, color=color, alpha=0.7, label=f"{phase}")
ax.set_yscale("symlog", linthresh=1.0)
ax.axhline(0.0, color="0.5", lw=0.8)
ax.set_xlabel("Your peak correlation coefficient")
ax.set_ylabel("Your dt minus PocketQuake's (ms)")
ax.set_title("Cross-correlation times of the Sangju pairs", fontsize=10)
fig.legend(loc="upper center", ncol=2, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig(Path(FIGURE_DIR) / "cc_times_vs_pocketquake.png", dpi=150)

### Your relocation against HypoDD's

HypoDD uses the layered model kim1983 and the **first arrival**: the direct wave or a head wave, whichever comes
first. For a source above 15 km, your straight rays at the top layer's speeds are exactly the direct wave.

**Watch out:** five of the six Sangju sources lie close to a boundary at 15 km. Beyond the crossover distance (M30)
the head wave along it arrives first, and straight rays cannot predict it.

Paste your `direct_time` and `head_wave` of M30 into the next cell and run it.

In [ ]:
# Paste your direct_time and head_wave of module M30 below this line.

The next cell turns the crossover distances of the deepest event above 15 km, rounded down, into station limits
for P and S. For the event below 15 km, it compares your `direct_time` with the straight ray. In HypoDD's printed
weighting, WDCT is the distance cutoff for catalog times (km).

In [ ]:
missing = [name for name in ("direct_time", "head_wave") if not callable(globals().get(name))]
if missing:
    raise NameError(f"{' and '.join(missing)} not found: paste your functions of module M30 into the cell above "
                    "and run it (finish task M30-T2 of module M30 first; task M30-T1 writes direct_time).")
MODEL = st.data.velocity_model(MODEL_NAME)
BOUNDARY_KM = float(MODEL["top_km"].iloc[1])                     # the boundary below the top layer
dtct_reloc = st.pq.read_reloc(PQ_SLUG, "dtct")
dtcc_reloc = st.pq.read_reloc(PQ_SLUG, "dtcc")
kept = [str(i) for i in dtct_reloc["id"] if i in set(dtcc_reloc["id"])]
print(f"HypoDD relocated {len(dtct_reloc)} (dt.ct) and {len(dtcc_reloc)} (dt.cc) of {len(events_dat)} events; "
      f"compared below: {kept}; left out by dt.ct: {sorted(set(events_dat['id']) - set(map(str, dtct_reloc['id'])))}")
inp_lines = (DD_DIR / "01.dt.ct" / "hypoDD.inp").read_text().splitlines()
at = next(n for n, line in enumerate(inp_lines) if line.startswith("*--- data weighting"))
print("hypoDD.inp (dt.ct):", inp_lines[at].strip())
print("\n".join(inp_lines[at + 1:at + 6]))

lat0, lon0 = events_dat["lat"].mean(), events_dat["lon"].mean()
start_rows = events_dat.set_index("id").loc[kept]
e0, n0 = flat_km(start_rows["lat"], start_rows["lon"], lat0, lon0)
start_km = np.column_stack([e0, n0, start_rows["depth_km"].to_numpy()])
se, sn = flat_km(stations_dat["lat"], stations_dat["lon"], lat0, lon0)
sites_km = np.column_stack([se, sn, np.zeros(len(se))])
reach = np.hypot(sites_km[:, None, 0] - start_km[None, :, 0], sites_km[:, None, 1] - start_km[None, :, 1])
far = reach.max(axis=1)                          # distance of each station from the farthest of the events
row_of = {e: i for i, e in enumerate(kept)}
site_of = {s: i for i, s in enumerate(stations_dat["station"])}
ct_kept = ct[ct["ev1"].isin(row_of) & ct["ev2"].isin(row_of)].reset_index(drop=True)
dd_obs = pd.DataFrame({"ev1": ct_kept["ev1"].map(row_of), "ev2": ct_kept["ev2"].map(row_of),
                       "sta": ct_kept["station"].map(site_of), "phase": ct_kept["phase"], "dt_s": ct_kept["dt_s"],
                       "weight": ct_kept["weight"] * np.where(ct_kept["phase"] == "P", 1.0, 0.5)})

depth_of = start_rows["depth_km"]
upper = depth_of[depth_of < BOUNDARY_KM]
print(f"\nHYPOINVERSE depths (km): {depth_of.round(2).to_dict()}; boundary of {MODEL_NAME} at {BOUNDARY_KM:g} km")
print(f"deepest event above the boundary: {upper.idxmax()} at {upper.max():.2f} km")
LIMIT_KM = {}
for phase in ("P", "S"):
    crossover = float(np.asarray(head_wave(np.array([50.0]), upper.max(), MODEL, phase, 1)[2]))
    LIMIT_KM[phase] = float(np.floor(crossover))
    used = dd_obs[(dd_obs["phase"] == phase) & (far[dd_obs["sta"].to_numpy()] <= LIMIT_KM[phase])]
    print(f"  {phase}: your crossover {crossover:.2f} km -> limit {LIMIT_KM[phase]:g} km; {used['sta'].nunique()} "
          f"stations with {len(used)} differential times inside it")
for event in depth_of[depth_of >= BOUNDARY_KM].index:
    for phase, speed in (("P", VP_KM_S), ("S", VS_KM_S)):
        pair_has = (dd_obs["ev1"] == row_of[event]) | (dd_obs["ev2"] == row_of[event])
        sta = np.unique(dd_obs.loc[pair_has & (dd_obs["phase"] == phase), "sta"].to_numpy())
        sta = sta[far[sta] <= LIMIT_KM[phase]]
        x = np.sort(reach[sta, row_of[event]])
        miss = 1000.0 * (direct_time(x, depth_of[event], MODEL, phase) - np.hypot(x, depth_of[event]) / speed)
        print(f"  event {event} at {depth_of[event]:.2f} km, {phase}: your layered direct time minus the straight "
              f"ray at its {len(x)} stations: {miss[0]:.1f} ms at the nearest ({x[0]:.0f} km), {miss[-1]:.1f} ms at "
              f"the farthest ({x[-1]:.0f} km), from {miss.min():.1f} to {miss.max():.1f} ms in all")

Your relocation of the events HypoDD kept (same start, `dt.ct` and weights), inside the limits and then with every
station. The cell compares shapes (positions minus their centroid) with HypoDD's.

In [ ]:
results = {}
for label, limits in (("inside the crossover", LIMIT_KM), ("all, to 100 km", {"P": np.inf, "S": np.inf})):
    inside = far[dd_obs["sta"].to_numpy()] <= dd_obs["phase"].map(limits).to_numpy()
    near = dd_obs[inside].reset_index(drop=True)
    found, shifts, rms = relocate(start_km, sites_km, near, VP_KM_S, VS_KM_S, DAMP, N_ITER, dd_system, dd_solve)
    results[label] = found
    print(f"\nyour relocation, stations {label}: {near['sta'].nunique()} stations, {len(near)} differential times; "
          f"weighted RMS by round (ms): {np.round(1000 * rms, 1).tolist()}")


def as_km(table):
    """x east, y north, z depth (km) of a relocation table, in the order of `kept`."""
    rows_ = table.set_index(table["id"].astype(str)).loc[kept]
    e, n = flat_km(rows_["lat"], rows_["lon"], lat0, lon0)
    return np.column_stack([e, n, rows_["depth"].to_numpy()])


def centred(xyz):
    return xyz - xyz.mean(axis=0)


sets = {"HYPOINVERSE": start_km, "HypoDD dt.ct": as_km(dtct_reloc), "HypoDD dt.cc": as_km(dtcc_reloc),
        **{f"yours, {k}": v for k, v in results.items()}}
print("\nshape differences from HypoDD dt.ct (RMS over the events, after removing each centroid; m):")
for label, xyz in sets.items():
    rms_m = 1000.0 * np.sqrt(np.mean((centred(xyz) - centred(sets["HypoDD dt.ct"])) ** 2, axis=0))
    shift_m = 1000.0 * (xyz.mean(axis=0) - start_km.mean(axis=0))
    print(f"  {label:>30}: east {rms_m[0]:6.1f}, north {rms_m[1]:6.1f}, depth {rms_m[2]:6.1f};   centroid minus "
          f"HYPOINVERSE's: {np.round(shift_m, 0)}")
per_event = pd.DataFrame(1000.0 * (centred(results["inside the crossover"]) - centred(sets["HypoDD dt.ct"])),
                         columns=["east_m", "north_m", "depth_m"], index=kept)
print("\nyours (inside the crossover) minus HypoDD dt.ct, per event, after removing each centroid (m):")
print(per_event.round(0).to_string())

below = [e for e in kept if depth_of[e] >= BOUNDARY_KM]
if below:                                        # the same relocation without the events below the boundary
    rest = [i for i, e in enumerate(kept) if e not in below]
    renumber = {old: new for new, old in enumerate(rest)}
    inside = far[dd_obs["sta"].to_numpy()] <= dd_obs["phase"].map(LIMIT_KM).to_numpy()
    near = dd_obs[inside & dd_obs["ev1"].isin(rest) & dd_obs["ev2"].isin(rest)]
    near = near.assign(ev1=near["ev1"].map(renumber), ev2=near["ev2"].map(renumber)).reset_index(drop=True)
    found, shifts, rms = relocate(start_km[rest], sites_km, near, VP_KM_S, VS_KM_S, DAMP, N_ITER, dd_system,
                                  dd_solve)
    print(f"\nyour relocation without {below}: {near['sta'].nunique()} stations, {len(near)} differential times; "
          f"weighted RMS by round (ms): {np.round(1000 * rms, 1).tolist()}")
    with_them = results["inside the crossover"][rest]
    for label, xyz in ((f"with {below}", with_them), (f"without {below}", found)):
        rms_m = 1000.0 * np.sqrt(np.mean((centred(xyz) - centred(sets["HypoDD dt.ct"][rest])) ** 2, axis=0))
        print(f"  the other {len(rest)} events, yours {label}: shape against HypoDD dt.ct east {rms_m[0]:.1f}, "
              f"north {rms_m[1]:.1f}, depth {rms_m[2]:.1f} m")
    change = 1000.0 * (centred(found) - centred(with_them))
    print(f"  leaving {below} out moved the others by at most {np.abs(change[:, :2]).max():.0f} m horizontally and "
          f"{np.abs(change[:, 2]).max():.0f} m in depth (after removing each centroid)")

fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.2))
styles = {"HYPOINVERSE": ("o", "0.6"), "HypoDD dt.ct": ("s", "tab:blue"), "HypoDD dt.cc": ("D", "tab:green"),
          "yours, inside the crossover": ("^", "tab:red")}
for label, (marker, color) in styles.items():
    xyz = sets[label] - start_km.mean(axis=0)
    axes[0].plot(1000 * xyz[:, 0], 1000 * xyz[:, 1], marker, color=color, ms=6, mfc="none",
                 label=label[0].upper() + label[1:])
    axes[1].plot(1000 * xyz[:, 0], 1000 * xyz[:, 2], marker, color=color, ms=6, mfc="none")
axes[0].set_xlabel("East of the HYPOINVERSE centroid (m)")
axes[0].set_ylabel("North of the HYPOINVERSE centroid (m)")
axes[0].set_aspect("equal", adjustable="datalim")
axes[0].set_title("Map view", fontsize=10)
axes[1].set_xlabel("East of the HYPOINVERSE centroid (m)")
axes[1].set_ylabel("Below the HYPOINVERSE centroid (m)")
axes[1].invert_yaxis()
axes[1].set_title("Depth section, looking north", fontsize=10)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=4, frameon=False, fontsize=8)
fig.tight_layout(rect=(0, 0, 1, 0.9))
fig.savefig(Path(FIGURE_DIR) / "relocations.png", dpi=150)

**Explain:** What changed when your relocation also used the stations beyond your crossover limits, and why?

## Part 7 · Reruns, the bootstrap and a colleague's file (about 90 min)

The **bootstrap** asks how a result would change if the data came out a little differently. PocketQuake redraws
the differential times with replacement and reruns HypoDD each time. For each event it reports the 95 % half-width:
half the spread of the middle 95 % of its positions (m). The next cell prints how it seeds each replica.

**The run log:** each run through `st.pq` adds a line to `work/.sgtl/runs.jsonl`. The line holds its start (`ts`,
UTC), its return code (0 = it worked) and the SHA-256 fingerprint of every file it wrote. A file's **modification
time** says when it was last written.

In [ ]:
source = (st.pq.pipeline_dir() / "pipeline" / "core" / "hypodd.py").read_text(encoding="utf-8").splitlines()
for number, line in enumerate(source, 1):
    if "Deterministic: replica" in line or "default_rng(seed + i)" in line or "Resampling is **global**" in line:
        print(f"hypodd.py:{number}: {line.strip()}")

### M41-T6 · Reruns, two bootstraps and a colleague's file · core · about 90 min

Write `who_wrote(path, log_path, slug)`. It returns the record of the latest run of cluster `slug` that logged the
SHA-256 of the file as it is now, or `None`. Then run the cells below and fill in `dd_runs`; its cell lists the
keys.

Example: a later failed run with empty `products` does not count.

In [ ]:
# A colleague's cell. They want to know whether HypoDD's catalog relocation depends on the number of cores, so they
# rerun the dt.ct stage on 8 cores and compare the relocation before and after. The run's own messages are hidden to
# keep their notebook short. Run it as it is.
colleague_before = st.pq.read_reloc(PQ_SLUG, "dtct")
colleague_started = UTCDateTime()
with contextlib.redirect_stdout(io.StringIO()):
    st.pq.run(PQ_SLUG, "dtct", "dtct", cores=8, mem_gb=PQ_MEM_GB, apply=True)
colleague_after = st.pq.read_reloc(PQ_SLUG, "dtct")
moved = (colleague_after[["x", "y", "z"]].to_numpy() - colleague_before[["x", "y", "z"]].to_numpy())
print(f"colleague: run started {colleague_started} (UTC); largest change of a relocated position: "
      f"{np.abs(moved).max():.1f} m")
print("colleague's verdict: HypoDD's catalog relocation does not depend on the number of cores.")

In [ ]:
# Before you code: should you trust the colleague's verdict, and why?
st.predict("M41-T6", """

""")

In [ ]:
def who_wrote(path, log_path, slug):
    """The most recent logged run of cluster `slug` whose products hold the SHA-256 of the file as it is now.

    path     : the product file (str or pathlib.Path)
    log_path : the run log in JSON Lines: one record per line, like those printed in Part 6; skip lines that are
               not valid JSON (empty or half-written) and records without products
    slug     : the cluster; runs of other clusters do not count
    returns  : the record (a dict, as json.loads reads it) of the last run of this cluster whose products (a dict
               of path: fingerprint) hold hashlib.sha256(<the file's bytes now>).hexdigest() among their values;
               match the bytes, not the file name; None when no run of this cluster logged that fingerprint
    """
    # Step 1: the SHA-256 fingerprint of the file's bytes (hashlib.sha256(...).hexdigest()).
    # Step 2: read the log line by line; skip lines that are not valid JSON (json.loads raises ValueError).
    # Step 3: keep the last record of this cluster whose products (a dict) have the fingerprint among their values.
    # YOUR CODE HERE
    raise NotImplementedError("M41-T6: write your code here")

Run 1: the dt.cc stage again, unchanged. Is HypoDD a **deterministic computation** (same input, same bytes)?

In [ ]:
st.pq.run(PQ_SLUG, "dtcc", "dtcc", cores=PQ_CORES, mem_gb=PQ_MEM_GB)            # dry run

In [ ]:
assert st.pq.run(PQ_SLUG, "dtcc", "dtcc", cores=PQ_CORES, mem_gb=PQ_MEM_GB, apply=True) == 0
rerun_copies = copy_dtcc_products("rerun")
print("copied the dt.cc products of the rerun:", [str(p) for p in rerun_copies])

Runs 2 and 3: two bootstraps. The cell copies `bootstrap_errors.csv` after each, since the second overwrites it.

In [ ]:
st.pq.bootstrap(PQ_SLUG, "dtcc", n=BOOT_N, seed=BOOT_SEED, cores=PQ_CORES, mem_gb=PQ_MEM_GB)     # dry run

In [ ]:
boot_copies = {}
for seed in (BOOT_SEED, BOOT_SEED_CHANGED):
    assert st.pq.bootstrap(PQ_SLUG, "dtcc", n=BOOT_N, seed=seed, cores=PQ_CORES, mem_gb=PQ_MEM_GB, apply=True) == 0
    boot_copies[seed] = Path(shutil.copy2(DD_DIR / "02.dt.cc" / "bootstrap_errors.csv",
                                          COPIES / f"bootstrap_seed{seed}.csv"))
    print("copied", boot_copies[seed])

In [ ]:
# Fill in dd_runs, a dict with exactly these keys:
#   "who_wrote"    your function
#   "slug"         PQ_SLUG
#   "first"        {path of each copy in first_copies (text): SHA-256 of that copy's bytes (hexdigest, 64 chars)}
#   "rerun"        the same for rerun_copies (the unchanged rerun)
#   "boot"         path of the bootstrap copy for BOOT_SEED (text)
#   "boot_changed" path of the copy for BOOT_SEED_CHANGED (text)
# The check recomputes every fingerprint from the file and finds it in the run log. Do not edit the copies. The two
# bootstraps must differ only in the seed, and the seeds must be at least BOOT_N apart.
# YOUR CODE HERE
raise NotImplementedError("M41-T6: write your code here")
print("copies in your report:", {k: (sorted(v) if isinstance(v, dict) else v) for k, v in dd_runs.items()
                                 if k in ("first", "rerun", "boot", "boot_changed")})

In [ ]:
st.check("M41-T6", answer=dd_runs)

What did the colleague read? Your `who_wrote` on the dt.ct `hypoDD.reloc`, its modification time, and the newest
log lines and files:

**Watch out:** a run that `st.pq` refuses writes nothing, so `hypoDD.reloc` stays the file of an older run: a
**stale file**.

In [ ]:
dtct_file = DD_DIR / "01.dt.ct" / "hypoDD.reloc"
writer = who_wrote(dtct_file, RUNS_LOG, PQ_SLUG)
print("the colleague's run started:   ", colleague_started)
if writer is None:
    print("no logged run of this cluster wrote the dt.ct relocation as it is now")
else:
    print(f"the run that wrote the file:   started {writer['ts']}, stages {writer.get('stage_from')} -> "
          f"{writer.get('through')}, cores {writer.get('cores')}, return code {writer.get('returncode')}")
print("the file was last modified:    ", UTCDateTime(dtct_file.stat().st_mtime))
print("\nlast lines of the run log (start, command, stages, cores, return code):")
for line in RUNS_LOG.read_text(encoding="utf-8").splitlines()[-4:]:
    run = json.loads(line)
    print(" ", run["ts"], run["command"], run.get("stage_from", run.get("branch")), "->", run.get("through", ""),
          "cores", run["cores"], "return code", run["returncode"])
print("\nnewest files in PocketQuake's log folder:", sorted(p.name for p in (RUN_DIR / "sgtl_logs").iterdir())[-4:])

The two bootstraps (half-widths in m), the centroid offset and the rerun's fingerprints:

In [ ]:
boots = {seed: pd.read_csv(path, comment="#").set_index("id") for seed, path in boot_copies.items()}
side = pd.concat({f"seed {seed}": table[["n_boot", "ex95", "ey95", "ez95"]] for seed, table in boots.items()},
                 axis=1)
print(side.round(1).to_string())
ratio = boots[BOOT_SEED_CHANGED][["ex95", "ey95", "ez95"]] / boots[BOOT_SEED][["ex95", "ey95", "ez95"]]
print(f"\nratio of the half-widths (seed {BOOT_SEED_CHANGED} / seed {BOOT_SEED}): from "
      f"{np.nanmin(ratio.to_numpy()):.2f} to {np.nanmax(ratio.to_numpy()):.2f}")
offset = 1000.0 * (sets["HypoDD dt.cc"].mean(axis=0) - start_km.mean(axis=0))
print(f"HypoDD dt.cc centroid minus HYPOINVERSE centroid (east, north, down; m): {np.round(offset, 0)}")
same = sorted(dd_runs["first"].values()) == sorted(dd_runs["rerun"].values())
print("the unchanged rerun wrote the same bytes (SHA-256 of every copy):", same)

Left: the bootstrap half-widths, which measure relative precision only (PocketQuake aligns the replicas). Right:
HypoDD's centroid minus HYPOINVERSE's, a difference between two methods. **Look:** can a tight cluster still have
an uncertain depth?

In [ ]:
from IPython.display import Image, display

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), constrained_layout=True)
visual_boot = boots[BOOT_SEED]
for column, name in (("ex95", "East"), ("ey95", "North"), ("ez95", "Depth")):
    axes[0].plot(visual_boot.index.astype(str), visual_boot[column], "o-", ms=4, label=name)
axes[0].set(xlabel="Event ID", ylabel="Aligned bootstrap 95% half-width (m)", title="Relative position uncertainty")
axes[0].tick_params(axis="x", rotation=45)
axes[0].legend(fontsize=8)
axes[1].bar(["East", "North", "Down"], offset, color=["tab:blue", "tab:orange", "0.5"])
axes[1].axhline(0, color="0.3", lw=0.7)
axes[1].set(ylabel="HypoDD minus HYPOINVERSE centroid (m)", title="Difference between methods, not known error")
fig.savefig(Path(FIGURE_DIR) / "M41_relative_and_common_position.png", dpi=150)
display(Image(filename=str(Path(FIGURE_DIR) / "M41_relative_and_common_position.png")))
plt.close(fig)

**Explain:** Did the colleague's run write the dt.ct relocation they read? Use the printed run log and modification
time.

## Wrap-up

Your results: `results/hypodd/`, `results/M41_cc_times.csv` and `figures/`. The records stayed raw counts, since a
correlation does not depend on the gain.

Station metadata: StationMetadata 1.0.2 by Byeong-Seok Ahn, Pukyong National University (`ahn2024`). KMA catalog:
Korea Open Government License Type 1 (attribution).

Your progress in this module:

In [ ]:
st.progress("M41")

**Optional:** bootstrap with seed 1 instead of `BOOT_SEED_CHANGED`. Why do the half-widths hardly change?

**Read more:** Waldhauser and Ellsworth (2000), the double-difference method; Waldhauser (2001), the HypoDD
manual; Schaff et al. (2004), sub-sample delays.

Next: M50 and M51 find the mechanisms of the Sangju earthquakes.